In [ ]:
!pip install mrmr_selection


In [ ]:
! pip install catboost

In [ ]:
pip install "transformers==5.0.0"

In [ ]:
pip install accelerate datasets torch

In [ ]:
pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

In [ ]:
pip install trl  sentencepiece peft accelerate bitsandbytes

In [ ]:
pip install torch torchvision

In [ ]:
pip install evaluate

In [ ]:
! nvidia-smi

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # only one GPU

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split,GridSearchCV
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost
from xgboost import XGBClassifier
from sklearn.linear_model import LinearRegression,LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from catboost import CatBoostClassifier 
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout,BatchNormalization, Activation
from sklearn.metrics import accuracy_score, confusion_matrix,classification_report

In [ ]:
data = pd.read_csv("/kaggle/input/datasets/ankushpanday1/alzheimers-prediction-dataset-global/alzheimers_prediction_dataset.csv")
data.head(5)

In [ ]:
print(f"shape of data is {data.shape}")
print(f"Columns are {data.columns}")

In [ ]:
data['Cholesterol Level'].value_counts()

In [ ]:
data['Genetic Risk Factor (APOE-ε4 allele)'].value_counts()

In [ ]:
data.isnull().sum()

In [ ]:
data.duplicated().sum()

In [ ]:
numerical_columns = data.select_dtypes(include=['int','float']).columns.tolist()
numerical_columns

In [ ]:
categorical_columns = data.select_dtypes(include= ['object']).columns.tolist()
categorical_columns

In [ ]:
data.describe(include='all')

In [ ]:
data.rename({'Alzheimer’s Diagnosis': 'Alzheimer_Diagnosis'},inplace=True)

In [ ]:
encoders = {}
for col in categorical_columns:
    lb = LabelEncoder()
    data[col] = lb.fit_transform(data[col].astype(str))
    encoders[col] = lb

In [ ]:
data.columns

In [ ]:
data.head(10)

In [ ]:
X = data.drop(columns=['Alzheimer’s Diagnosis'],axis=1)
y= data['Alzheimer’s Diagnosis']

In [ ]:
X.head(3)

In [ ]:
y.value_counts()

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)

In [ ]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_resample,y_train_resample = smote.fit_resample(X_train,y_train)

In [ ]:
X_train = X_train_resample
y_train = y_train_resample

In [ ]:
y_train.value_counts()

In [ ]:
rf = RandomForestClassifier(n_estimators=200,random_state=42)
rf.fit(X_train,y_train)

In [ ]:
from lime.lime_tabular import LimeTabularExplainer
X_lime = X_test.sample(n=100,random_state=42)
explainer = LimeTabularExplainer(
    X_lime.values,
    feature_names = X_lime.columns,
    class_names= ["No", "Yes"],
    mode = "classification",
    random_state=42
    
)

In [ ]:
from collections import defaultdict
lime_importance = defaultdict(list)
lime_scores = np.zeros(X_lime.shape[1])

for i in range(len(X_lime)):
    explanation = explainer.explain_instance(
    X_lime.iloc[i].values,
    rf.predict_proba,
    num_features = 10
    )
    for features,weight in explanation.local_exp[1]:
        lime_scores[features] += abs(weight)
        

lime_scores = lime_scores/len(X_lime)  
lime_importance = pd.Series(
    lime_scores,
    index = X_lime.columns,
    
).sort_values(ascending=False)

top_three_lime = lime_importance.head(3)
print(f"Top 3 Features:{top_three_lime}")
print("\n")
top_five_lime = lime_importance.head(5)
print(f"Top 5 Features:{top_five_lime}")
print("\n")
top_ten_lime = lime_importance.head(10)
print(f"Top 10 Features:{top_ten_lime}")


# SHAP Features Selections

In [ ]:
import shap
X_shap = X_test.sample(n=100,random_state=42)
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_shap)
shap_value_yes = shap_values[1]
shap_importance = np.abs(shap_value_yes).mean(axis=1)

In [ ]:
shap_importance = pd.Series(
    shap_importance,
    index = X_shap.columns,
    
).sort_values(ascending=False)

top_three_shap = shap_importance.head(3)
print(f"Top 3 Features:{top_three_shap}")
print("\n")
top_five_shap = shap_importance.head(5)
print(f"Top 5 Features: {top_five_shap}")
print("\n")
top_ten_shap =shap_importance.head(10)
print(f"Top 10 Features:{top_ten_shap}")

In [ ]:
def jacard_similarity(set1,set2):
    set1 = set(set1)
    set2 = set(set2)
    A = len(set1.intersection(set2))
    B = len(set1.union(set2))
    return A/B

shap_features = top_ten_shap.index
lime_features = top_ten_lime.index
similarity_score = jacard_similarity(shap_features,lime_features)
print(f"similarity score for top 10 Features is:{similarity_score}")

In [ ]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="median")

X_train = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

In [ ]:
from mrmr import mrmr_classif

selected_features = mrmr_classif(
    X=X_train,
    y=y_train,
    K=10
)

print("Selected features:")
print(selected_features)

In [ ]:
top_ten_mmr = selected_features
print(top_ten_mmr)

In [ ]:
top_ten_shap.index

In [ ]:
top_ten_lime.index

In [ ]:
top_ten_RFE

# Feature Selection using RFE

In [ ]:
from sklearn.feature_selection import RFE 
from lightgbm import LGBMClassifier

model = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42,
    verbosity=-1
)

rfe = RFE(
    estimator = model,
    n_features_to_select = 10,
    step =1
)

rfe.fit(X_train,y_train)
selected_column = X_train.columns[rfe.support_]
print("Top 10 selected features")
top_ten_RFE= selected_column.to_list()


In [ ]:
top_ten_RFE= selected_column.to_list()
print(top_ten_RFE)

# Jacard Similary between MMR and RFE feature Selection 

In [ ]:
top_ten_mmr

In [ ]:
 top_ten_RFE

In [ ]:
def jacard_similarity(set1,set2):
    set1 = set(set1)
    set2 = set(set2)
    A = len(set1.intersection(set2))
    B = len(set1.union(set2))
    return A/B

shap_features = top_ten_mmr
lime_features = top_ten_RFE
similarity_score = jacard_similarity(shap_features,lime_features)
print(f"similarity score for top 10 Features is:{similarity_score}")

In [ ]:
def jacard_similarity(set1,set2):
    set1 = set(set1)
    set2 = set(set2)
    A = len(set1.intersection(set2))
    B = len(set1.union(set2))
    return A/B

shap_features = top_ten_shap.index
lime_features = top_ten_mmr
similarity_score = jacard_similarity(shap_features,lime_features)
print(f"similarity score for top 10 Features is:{similarity_score}")

In [ ]:
def jacard_similarity(set1,set2):
    set1 = set(set1)
    set2 = set(set2)
    A = len(set1.intersection(set2))
    B = len(set1.union(set2))
    return A/B

shap_features = top_ten_lime.index
lime_features = top_ten_mmr
similarity_score = jacard_similarity(shap_features,lime_features)
print(f"similarity score for top 10 Features is:{similarity_score}")

# Top 10 Features 

In [ ]:
from collections import Counter

list1 = [
    'Age', 'Genetic Risk Factor (APOE-ε4 allele)',
    'Family History of Alzheimer’s', 'Stress Levels',
    'Urban vs Rural Living', 'Sleep Quality', 'Gender',
    'Employment Status', 'Air Pollution Exposure', 'Hypertension'
]

list2 = [
    'Age', 'Stress Levels',
    'Genetic Risk Factor (APOE-ε4 allele)',
    'Urban vs Rural Living', 'Sleep Quality',
    'Employment Status', 'Family History of Alzheimer’s',
    'Dietary Habits', 'Depression Level', 'Physical Activity Level'
]

list3 = [
    'Age', 'Genetic Risk Factor (APOE-ε4 allele)',
    'Family History of Alzheimer’s', 'Urban vs Rural Living',
    'Hypertension', 'Diabetes', 'Gender',
    'Alcohol Consumption', 'Depression Level',
    'Employment Status'
]
list4 = ['Country',
 'Age',
 'Education Level',
 'BMI',
 'Cognitive Test Score',
 'Depression Level',
 'Sleep Quality',
 'Air Pollution Exposure',
 'Employment Status',
 'Stress Levels']

counts = Counter(list1+list2+list3+list4)
top_10 =counts.most_common(10)
for feature,frequency in top_10:
    print(f"Features are {feature} and Freaquency is {frequency}")
    

# Using ANN

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
selected_features= [
    'Age',
    'Employment Status',
    'Genetic Risk Factor (APOE-ε4 allele)',
    'Family History of Alzheimer’s',
    'Stress Levels',
    'Urban vs Rural Living',
    'Sleep Quality',
    'Depression Level',
    'Gender',
    'Air Pollution Exposure'
    
    
    
]
for col in categorical_columns:
    lb = LabelEncoder()
    data[col] = lb.fit_transform(data[col])
X = data[selected_features]
y = data['Alzheimer’s Diagnosis']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2, random_state=42)
smote = SMOTE(random_state = 42)
X_train_resample,y_train_resample = smote.fit_resample(X_train,y_train)
X_train = X_train_resample
y_train = y_train_resample
model = Sequential([
    Dense(192,input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Activation("relu"),
    Dropout(0.3),
    Dense(128),
    BatchNormalization(),
    Activation("relu"),
    Dropout(0.3),
    Dense(64),
    BatchNormalization(),
    Activation("relu"),
    Dropout(0.3),
    Dense(1,activation="sigmoid")
])
model.compile(optimizer = "adam",loss="binary_crossentropy", metrics= ["accuracy"])
earstopping = EarlyStopping(monitor="val_loss",patience=15,restore_best_weights=True)
history = model.fit(X_train,y_train,batch_size=32,epochs=50,validation_split=0.2,callbacks=[earstopping])

In [ ]:
loss,accuracy = model.evaluate(X_test,y_test)
print(f"Accuracy is {accuracy*100}%")
predict= (model.predict(X_test)>0.5).astype("int")
print(classification_report(y_test, predict))
print(confusion_matrix(y_test, predict))

# XGBOOST CLASSIFIER

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
selected_features= [
    'Age',
    'Employment Status',
    'Genetic Risk Factor (APOE-ε4 allele)',
    'Family History of Alzheimer’s',
    'Stress Levels',
    'Urban vs Rural Living',
    'Sleep Quality',
    'Depression Level',
    'Gender',
    'Smoking Status'
     
    
]
for col in categorical_columns:
    lb = LabelEncoder()
    data[col] = lb.fit_transform(data[col])
X = data[selected_features]
y = data['Alzheimer’s Diagnosis']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2, random_state=42)
smote = SMOTE(random_state = 42)
X_train_resample,y_train_resample = smote.fit_resample(X_train,y_train)
X_train = X_train_resample
y_train = y_train_resample
sc = StandardScaler()
X_train_sc= sc.fit_transform(X_train)
X_test_sc = sc.transform(X_test)

xgb = XGBClassifier(
    n_estimators=300,max_depth=5,learning_rate=0.005,random_state=42
)
xgb.fit(X_train_sc,y_train)
predict_xgb = xgb.predict(X_test_sc)
accuracy = accuracy_score(y_test,predict_xgb)
print(f"Accuracy Score For XGBoost Classifier:{accuracy*100}%")
print("Train Accuracy:", xgb.score(X_train_sc, y_train))
print("Test Accuracy:", xgb.score(X_test_sc, y_test))
print(classification_report(y_test, predict_xgb))
print(confusion_matrix(y_test, predict_xgb))

# XGBoost with Different Features with Accuracy 73%

In [ ]:
from sklearn.metrics import accuracy_score
selected_features= [
    'Age', 
    'Genetic Risk Factor (APOE-ε4 allele)', 
    'Family History of Alzheimer’s', 
    'Education Level',
    'Country',
    'Income Level', 
    'Urban vs Rural Living', 
    'Stress Levels',
    'Smoking Status',
    'Employment Status',
   
    
]

X = data[selected_features]
y = data['Alzheimer’s Diagnosis']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)
X_train_resample,y_train_resample = smote.fit_resample(X_train,y_train)
X_train = X_train_resample
y_train = y_train_resample

sc = StandardScaler()
X_train_sc= sc.fit_transform(X_train)
X_test_sc = sc.transform(X_test)

xgb = XGBClassifier(
    n_estimators=300,max_depth=5,learning_rate=0.005,random_state=42
)
xgb.fit(X_train_sc,y_train)
predict_xgb = xgb.predict(X_test_sc)
accuracy = accuracy_score(y_test,predict_xgb)
print(f"Accuracy Score For XGBoost Classifier:{accuracy*100}%")
print("Train Accuracy:", xgb.score(X_train_sc, y_train))
print("Test Accuracy:", xgb.score(X_test_sc, y_test))
print(classification_report(y_test, predict_xgb))
print(confusion_matrix(y_test, predict_xgb))

# MLP : Multi Layer Perceptron

In [ ]:
selected_features= [
    'Age',
    'Employment Status',
    'Genetic Risk Factor (APOE-ε4 allele)',
    'Family History of Alzheimer’s',
    'Stress Levels',
    'Urban vs Rural Living',
    'Sleep Quality',
    'Depression Level',
    'Gender',
    'Air Pollution Exposure'
     
    
]
for col in categorical_columns:
    lb = LabelEncoder()
    data[col] = lb.fit_transform(data[col])
X = data[selected_features]
y = data['Alzheimer’s Diagnosis']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2, random_state=42)
smote = SMOTE(random_state = 42)
X_train_resample,y_train_resample = smote.fit_resample(X_train,y_train)
X_train = X_train_resample
y_train = y_train_resample

sc = StandardScaler()
X_train_sc= sc.fit_transform(X_train)
X_test_sc = sc.transform(X_test)



mlp = MLPClassifier(
    hidden_layer_sizes = (128,64,32),
    activation = "relu",
    solver = "adam",
    alpha = 1e-3,
    max_iter = 800,
    early_stopping = True,
    random_state=42
)
mlp.fit(X_train_sc,y_train)
predict_mlp = mlp.predict(X_test_sc)
accuracy = accuracy_score(y_test,predict_mlp)
print(f"Accuracy Score For MLP Classifier:{accuracy*100}%")
print(classification_report(y_test, predict_mlp))
print(confusion_matrix(y_test, predict_mlp))

# CatBoost

In [ ]:
selected_features= [
    'Age',
    'Employment Status',
    'Genetic Risk Factor (APOE-ε4 allele)',
    'Family History of Alzheimer’s',
    'Stress Levels',
    'Urban vs Rural Living',
    'Sleep Quality',
    'Depression Level',
    'Gender',
    'Air Pollution Exposure'
     
    
]
for col in categorical_columns:
    lb = LabelEncoder()
    data[col] = lb.fit_transform(data[col])
X = data[selected_features]
y = data['Alzheimer’s Diagnosis']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2, random_state=42)
smote = SMOTE(random_state = 42)
X_train_resample,y_train_resample = smote.fit_resample(X_train,y_train)
X_train = X_train_resample
y_train = y_train_resample

sc = StandardScaler()
X_train_sc= sc.fit_transform(X_train)
X_test_sc = sc.transform(X_test)

catboost_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100
)

catboost_model.fit(X_train_sc,y_train)
predict_cat = catboost_model.predict(X_test_sc)
accuracy = accuracy_score(y_test,predict_cat)
print(f"Accuracy Score For MLP Classifier:{accuracy*100}%")
print(classification_report(y_test, predict_cat))
print(confusion_matrix(y_test, predict_cat))

In [ ]:
for col in categorical_columns:
    data[col] =encoders[col].inverse_transform(data[col])

In [ ]:
data.head(5)

In [ ]:
selected_features= [
    'Age',
    'Employment Status',
    'Genetic Risk Factor (APOE-ε4 allele)',
    'Family History of Alzheimer’s',
    'Stress Levels',
    'Urban vs Rural Living',
    'Sleep Quality',
    'Depression Level',
    'Gender',
    'Air Pollution Exposure',
    'Alzheimer’s Diagnosis'
     
    
]
data = data[selected_features]
data.head(4)

In [ ]:
train_df,test_df = train_test_split(
    data,
    test_size=0.2,
    random_state=42,
    stratify = data[target]
)

In [ ]:
jsondata = []
for _, row in data.iterrows():
    features = row.drop(target).to_dict()
    jsondata.append({
        "input": features,
        "prediction": row[target].item()
    })

In [ ]:
test_label = [item['prediction'] for item in jsondata]

In [ ]:
import json
with open("/kaggle/working/data.json","w") as file:
    json.dump(jsondata,file,indent=2)

In [ ]:
with open("/kaggle/working/data.json","r") as file:
    data = json.load(file)

In [ ]:
'Age',
    'Employment Status',
    'Genetic Risk Factor (APOE-ε4 allele)',
    'Family History of Alzheimer’s',
    'Stress Levels',
    'Urban vs Rural Living',
    'Sleep Quality',
    'Depression Level',
    'Gender',
    'Air Pollution Exposure',
    'Alzheimer’s Diagnosis'

In [ ]:
rows = []
for patient in data:
    text = (
        f"The patient's age is {patient['input']['Age']}."
        f"The patient's Employment Status is {patient['input']['Employment Status']}."
        f"The patient's Genetic Risk Factor (APOE-ε4 allele) is {patient['input']['Genetic Risk Factor (APOE-ε4 allele)']}."
        f"The patient's Family History of Alzheimer’s is {patient['input']['Family History of Alzheimer’s']}."
        f"The patient's Stress Levels is {patient['input']['Stress Levels']}."
        f"The patient's Urban vs Rural Living is {patient['input']['Urban vs Rural Living']}."
        f"The patient's Sleep Quality is {patient['input']['Sleep Quality']}."
        f"The patient's Depression Level is {patient['input']['Depression Level']}."
        f"The patient's Depression Level is {patient['input']['Gender']}."
        f"The patient's Air Pollution Exposure is {patient['input']['Air Pollution Exposure']}."
    )

    target = patient['prediction']
    rows.append({"text": text,"Alzheimer’s Diagnosis": target})
    

In [ ]:
data = pd.DataFrame(rows, columns=['text','Alzheimer’s Diagnosis'])

In [ ]:
data.to_csv("alzhermer_text_data.csv",index=False)

In [ ]:
data = pd.read_csv("/kaggle/working/alzhermer_text_data.csv")

data['text'][10]